# Overture — Places quickstart (live)

Fetch points of interest for a small bbox and read them back. Overture Places is public and anonymous — no credentials.

## Setup

A single import of the unified `EarthLens` entry point. Overture is a vector backend that writes GeoParquet, so this quickstart pairs it with `geopandas` later to read the result back.

In [ ]:
from earthlens.core import EarthLens

## Define the area of interest

A tiny bounding box over one Times Square block, plus an output directory for the GeoParquet file.

In [ ]:
LAT_LIM = [40.757, 40.759]  # [south, north]
LON_LIM = [-73.987, -73.984]  # [west, east] — a Times Square block
OUT = "_overture_out"

## Build the request

Configure the Overture backend for the `places` dataset over our bbox. Passing `variables=[]` selects the theme primary type (`place`).

In [ ]:
places = EarthLens(
    data_source="overture",
    dataset="places",
    variables=[],  # [] -> the theme primary type, "place"
    aoi=[LON_LIM[0], LAT_LIM[0], LON_LIM[1], LAT_LIM[1]],
    path=OUT,
)

## Download the features

`download()` writes the matching places to a GeoParquet file under `OUT` and returns the list of written paths.

In [ ]:
paths = places.download()
paths

## Read the GeoParquet back

Load the written file with `geopandas` and confirm the feature count and CRS. Each row carries a per-feature `license_id`.

In [ ]:
import geopandas as gpd

gdf = gpd.read_parquet(paths[0])
print(len(gdf), 'places;  CRS', gdf.crs.to_epsg())
gdf[['id', 'license_id']].head()

## Pull out the place names

Place names live in the nested `names` struct; extract the primary name for a quick, readable look at what's in the block.

In [ ]:
def primary_name(names):
    if isinstance(names, dict):
        return names.get('primary')
    return None


gdf['name'] = gdf['names'].apply(primary_name)
gdf[['name', 'license_id']].dropna(subset=['name']).head(10)